In [1]:
import os
import sys
import plotly.express as px
import logging
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, Javascript
sys.path.append("../../../..")
sys.path.append("../../../../scripts")
sys.path.append("../../../../scripts/summarize/calibration")

from notebooks.notebook_styling import bellevue_theme
from input_configuration import *
from h5toDF import *
from summary_functions import *
from dictionary import *
from utils import survey_year, get_subarea, get_data

logging.disable(logging.CRITICAL)

In [2]:
# data processing
taz_subarea = pd.read_csv(os.path.join(project_folder, districtfile))
taz_subarea['DistrictFlowName'] = taz_subarea['DistrictFlowID'].map(district_flow_name)
taz_subarea.rename(columns={'BKRCastTAZ': 'TAZ'}, inplace=True)
data_daysim = convert(os.path.join(project_folder, h5_results_file), 
                      os.path.join(project_folder, guidefile), 
                      os.path.join(project_folder, h5_results_name), stdout=False)
data_survey = convert(os.path.join(project_folder, h5_comparison_file),   # data_survey and data_fullsurvey are different in 2023, data_survey has a smaller number of records
                      os.path.join(project_folder, guidefile), 
                      os.path.join(project_folder, h5_comparison_name), stdout=False)
data_fullsurvey = convert(os.path.join(project_folder, h5_fullsurvey_file), 
                          os.path.join(project_folder, guidefile), 
                          os.path.join(project_folder, h5_fullsurvey_name), stdout=False)

data_survey['Trip_cloned']['mode'] = data_survey['Trip_cloned']['mode'].replace('TNC','Other')
data_fullsurvey['Trip']['mode'] = data_fullsurvey['Trip']['mode'].replace('TNC','Other')

# locate the ACS survey data
acs_data = os.path.join(project_folder, f'inputs/model/survey/ACS_2023.xlsx')
acs_data_bkr = os.path.join(project_folder, f'inputs/model/survey/ACS_2023_BKR.xlsx')

## Total Tours

In [ ]:
def total_tours(data1, data2, tag='PSRC Region'):
    Tour_1_total = get_total(data1['Tour']['toexpfac'])
    Tour_2_total = get_total(data2['Tour_cloned']['toexpfac'])

    if int(model_year) >= 2023:
        # because in year 2023, many records got dropped after converting into tours.
        # trip weights before converting into tours is 1.133089 times of the trip weights after conversion.
        # we use this scale to recover back the tour weights
        Tour_2_total *= 1.133089

    tpp  = pd.DataFrame(index = ['Tours'])
    tpp['DaysimOutputs'] = Tour_1_total
    tpp[f'{survey_year}Survey'] = Tour_2_total
    tpp = get_differences(tpp, 'DaysimOutputs', f'{survey_year}Survey', 2)
    # table
    display(tpp.style.format({
        'DaysimOutputs': '{:,.1f}',
        f'{survey_year}Survey': '{:,.1f}',
        f"Difference (DaysimOutputs - {survey_year}Survey)": '{:,.1f}',
        f"% Difference (DaysimOutputs - {survey_year}Survey)": '{:,.1f}%',}))

In [4]:
total_tours(data1=data_daysim, data2=data_survey, tag='PSRC Region')

,DaysimOutputs,2023Survey,Difference (DaysimOutputs - 2023Survey),% Difference (DaysimOutputs - 2023Survey)
Tours,"5,875,487.0","5,050,722.0","824,764.9",16.3%


In [5]:
import copy
_data_daysim = copy.deepcopy(data_daysim)
_data_survey = copy.deepcopy(data_survey)
_data_fullsurvey = copy.deepcopy(data_fullsurvey)
fname_tail, data_daysim_bkr, data_survey_bkr, data_fullsurvey_bkr = \
    get_data(data1=_data_daysim, data2=_data_survey, data3=_data_fullsurvey, taz_subarea=taz_subarea, if_region=False)
total_tours(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

,DaysimOutputs,2023Survey,Difference (DaysimOutputs - 2023Survey),% Difference (DaysimOutputs - 2023Survey)
Tours,"462,173.0","332,686.9","129,486.1",38.9%


## Tour per Person

In [6]:
def tour_per_person(data1, data2, tag='PSRC Region'):
    Person_1_total = get_total(data1['Person']['psexpfac'])
    Person_2_total = get_total(data2['Person']['psexpfac'])
    Tour_1_total = get_total(data1['Tour']['toexpfac'])
    Tour_2_total = get_total(data2['Tour_cloned']['toexpfac'])

    ##Tours per person
    tpp1 = Tour_1_total / Person_1_total
    tpp2 = Tour_2_total / Person_2_total
    tpp  = pd.DataFrame(index = ['Tours'])
    tpp['DaysimOutputs'] = tpp1
    tpp[f'{survey_year}Survey'] = tpp2
    tpp = get_differences(tpp, 'DaysimOutputs', f'{survey_year}Survey', 2)
    # table
    display(tpp.style.format({
        'DaysimOutputs': '{:,.1f}',
        f'{survey_year}Survey': '{:,.1f}',
        f"Difference (DaysimOutputs - {survey_year}Survey)": '{:,.1f}',
        f"% Difference (DaysimOutputs - {survey_year}Survey)": '{:,.1f}%',}))

In [7]:
tour_per_person(data1=data_daysim, data2=data_survey, tag='PSRC Region')

,DaysimOutputs,2023Survey,Difference (DaysimOutputs - 2023Survey),% Difference (DaysimOutputs - 2023Survey)
Tours,1.4,1.3,0.0,2.2%


In [8]:
tour_per_person(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

,DaysimOutputs,2023Survey,Difference (DaysimOutputs - 2023Survey),% Difference (DaysimOutputs - 2023Survey)
Tours,1.4,1.1,0.3,23.8%


## Tour Share by Purpose

In [9]:
def pc_tour_by_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region'):
    ##Percent of Tours by Purpose
    tour_1_total = get_total(data_daysim['Tour']['toexpfac'])
    tour_2_total = get_total(data_survey['Tour_cloned']['toexpfac'])
    ptbp1 = 100 * data1['Tour'][['pdpurp','toexpfac']].groupby('pdpurp').sum()['toexpfac'] / tour_1_total
    ptbp2 = 100 * data2['Tour_cloned'][['pdpurp','toexpfac']].groupby('pdpurp').sum()['toexpfac'] / tour_2_total
    ptbp = pd.DataFrame()
    ptbp['Percent of Tours (DaysimOutputs)'] = ptbp1
    ptbp[f'Percent of Tours ({survey_year}Survey)'] = ptbp2
    ptbp = get_differences(ptbp,'Percent of Tours (DaysimOutputs)', f'Percent of Tours ({survey_year}Survey)', 2)
    ptbp = recode_index(ptbp, 'pdpurp', 'Tour Purpose')
    ptbp = ptbp.loc[pdpurp_cat.values()]
    # table
    display(ptbp.style.format({
        'Percent of Tours (DaysimOutputs)': '{:,.1f}%',
        f'Percent of Tours ({survey_year}Survey)': '{:,.1f}%',
        f"Difference (Percent of Tours (DaysimOutputs) - Percent of Tours ({survey_year}Survey))": '{:,.1f}%',
        f"% Difference (Percent of Tours (DaysimOutputs) - Percent of Tours ({survey_year}Survey))": '{:,.1f}%',
    }))
    # plot
    fig = px.bar(
        ptbp.reset_index(),
        x='Tour Purpose',
        y=['Percent of Tours (DaysimOutputs)', f'Percent of Tours ({survey_year}Survey)'],
        barmode='group',
        title=f'Percent of Tours by Purpose ({tag})'
    )
    fig.update_layout(yaxis_title='Percent of Tours', 
                      xaxis_title='Tour Purpose',
                      xaxis=dict(showgrid=True), 
                      yaxis=dict(showgrid=True))
    fig.update_yaxes(ticksuffix='%')
    fig.show()

In [10]:
pc_tour_by_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region')

,Percent of Tours (DaysimOutputs),Percent of Tours (2023Survey),Difference (Percent of Tours (DaysimOutputs) - Percent of Tours (2023Survey)),% Difference (Percent of Tours (DaysimOutputs) - Percent of Tours (2023Survey))
Tour Purpose,,,,
Work,27.6%,27.5%,0.0%,0.1%
School,9.8%,9.7%,0.1%,0.7%
Escort,11.1%,10.8%,0.3%,2.5%
Personal Business,7.0%,6.8%,0.2%,2.4%
Shop,12.9%,12.9%,-0.0%,-0.1%
Meal,7.1%,7.0%,0.1%,2.1%
Social,24.6%,25.3%,-0.7%,-2.7%


In [11]:
pc_tour_by_purp(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

,Percent of Tours (DaysimOutputs),Percent of Tours (2023Survey),Difference (Percent of Tours (DaysimOutputs) - Percent of Tours (2023Survey)),% Difference (Percent of Tours (DaysimOutputs) - Percent of Tours (2023Survey))
Tour Purpose,,,,
Work,1.9%,1.8%,0.1%,7.7%
School,0.9%,0.8%,0.1%,7.0%
Escort,0.8%,0.8%,0.1%,11.8%
Personal Business,0.5%,0.3%,0.2%,81.2%
Shop,0.8%,0.5%,0.3%,55.5%
Meal,0.7%,0.4%,0.2%,54.2%
Social,2.2%,2.0%,0.2%,12.1%


## Tour Share by Mode

In [12]:
def pc_tour_by_mode(data1=data_daysim, data2=data_survey, tag='PSRC Region'):
    ##Percent of Tours by Mode
    tour_1_total = get_total(data_daysim['Tour']['toexpfac'])
    tour_2_total = get_total(data_survey['Tour_cloned']['toexpfac'])
    ptbp1 = 100 * data1['Tour'][['tmodetp','toexpfac']].groupby('tmodetp').sum()['toexpfac'] / tour_1_total
    ptbp2 = 100 * data2['Tour_cloned'][['tmodetp','toexpfac']].groupby('tmodetp').sum()['toexpfac'] / tour_2_total
    ptbp = pd.DataFrame()
    ptbp['Percent of Tours (DaysimOutputs)'] = ptbp1
    ptbp[f'Percent of Tours ({survey_year}Survey)'] = ptbp2
    ptbp = get_differences(ptbp,'Percent of Tours (DaysimOutputs)', f'Percent of Tours ({survey_year}Survey)', 2)
    ptbp = recode_index(ptbp, 'tmodetp', 'Tour Mode')
    ptbp = ptbp.loc[mode_cat.values()]
    # table
    display(ptbp.style.format({
        'Percent of Tours (DaysimOutputs)': '{:,.1f}%',
        f'Percent of Tours ({survey_year}Survey)': '{:,.1f}%',
        f"Difference (Percent of Tours (DaysimOutputs) - Percent of Tours ({survey_year}Survey))": '{:,.1f}%',
        f"% Difference (Percent of Tours (DaysimOutputs) - Percent of Tours ({survey_year}Survey))": '{:,.1f}%',
    }))
    # plot
    fig = px.bar(
        ptbp.reset_index(),
        x='Tour Mode',
        y=['Percent of Tours (DaysimOutputs)', f'Percent of Tours ({survey_year}Survey)'],
        barmode='group',
        title=f'Tour Share by Mode ({tag})'
    )
    fig.update_layout(yaxis_title='Tour Share', 
                      xaxis_title='Tour Mode',
                      xaxis=dict(showgrid=True), 
                      yaxis=dict(showgrid=True))
    fig.update_yaxes(ticksuffix='%')
    fig.show()

In [13]:
pc_tour_by_mode(data1=data_daysim, data2=data_survey, tag='PSRC Region')

,Percent of Tours (DaysimOutputs),Percent of Tours (2023Survey),Difference (Percent of Tours (DaysimOutputs) - Percent of Tours (2023Survey)),% Difference (Percent of Tours (DaysimOutputs) - Percent of Tours (2023Survey))
Tour Mode,,,,
Walk,10.8%,10.9%,-0.2%,-1.4%
Bike,0.6%,1.2%,-0.7%,-55.4%
SOV,38.6%,37.3%,1.3%,3.6%
HOV2,22.4%,22.8%,-0.4%,-1.8%
HOV3+,21.6%,19.5%,2.1%,10.8%
Transit Walk Access,3.5%,4.3%,-0.8%,-19.4%
Transit Auto Access,0.2%,0.4%,-0.2%,-49.6%
School Bus,2.4%,2.5%,-0.1%,-3.0%


In [14]:
pc_tour_by_mode(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

,Percent of Tours (DaysimOutputs),Percent of Tours (2023Survey),Difference (Percent of Tours (DaysimOutputs) - Percent of Tours (2023Survey)),% Difference (Percent of Tours (DaysimOutputs) - Percent of Tours (2023Survey))
Tour Mode,,,,
Walk,0.6%,0.8%,-0.2%,-28.2%
Bike,0.0%,0.1%,-0.0%,-52.2%
SOV,3.0%,2.2%,0.8%,38.5%
HOV2,1.9%,1.6%,0.3%,17.5%
HOV3+,1.6%,1.1%,0.5%,48.1%
Transit Walk Access,0.5%,0.3%,0.2%,64.6%
Transit Auto Access,0.0%,0.2%,-0.1%,-87.4%
School Bus,0.2%,0.2%,0.0%,2.2%


## Tours per Person by Purpose

In [15]:
def tours_per_ps_by_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region'):
    ##Tours per Person by Purpose
    Person_1_total = get_total(data1['Person']['psexpfac'])
    Person_2_total = get_total(data2['Person']['psexpfac'])
    tpbp1 = data1['Tour'][['pdpurp','toexpfac']].groupby('pdpurp').sum()['toexpfac'] / Person_1_total
    tpbp2 = data2['Tour_cloned'][['pdpurp','toexpfac']].groupby('pdpurp').sum()['toexpfac'] / Person_2_total
    tpbp = pd.DataFrame()
    tpbp['Tours per Person (DaysimOutputs)'] = tpbp1
    tpbp[f'Tours per Person ({survey_year}Survey)'] = tpbp2
    tpbp = get_differences(tpbp, 'Tours per Person (DaysimOutputs)', 'Tours per Person ({survey_year}Survey)', 2)
    tpbp = recode_index(tpbp, 'pdpurp', 'Tour Purpose')
    tpbp = tpbp.loc[pdpurp_cat.values()]
    # table
    display(tpbp.style.format({
        'Tours per Person (DaysimOutputs)': '{:,.1f}',
        f'Tours per Person ({survey_year}Survey)': '{:,.1f}',
        f"Difference (Tours per Person (DaysimOutputs) - Tours per Person ({survey_year}Survey))": '{:,.1f}',
        f"% Difference (Tours per Person (DaysimOutputs) - Tours per Person ({survey_year}Survey))": '{:,.1f}%',
    }))
    # plot
    fig = px.bar(
        tpbp.reset_index(),
        x='Tour Purpose',
        y=['Tours per Person (DaysimOutputs)', f'Tours per Person ({survey_year}Survey)'],
        barmode='group',
        title=f'Tours per Person by Purpose ({tag})'
    )
    fig.update_layout(yaxis_title='Tours per Person', 
                      xaxis_title='Tour Purpose', 
                      xaxis=dict(showgrid=True), 
                      yaxis=dict(showgrid=True))
    fig.show()    

In [16]:
tours_per_ps_by_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region')

,Tours per Person (DaysimOutputs),Tours per Person (2023Survey),Difference (Tours per Person (DaysimOutputs) - Tours per Person (2023Survey)),% Difference (Tours per Person (DaysimOutputs) - Tours per Person (2023Survey))
Tour Purpose,,,,
Work,0.4,0.4,0.0,2.3%
School,0.1,0.1,0.0,2.8%
Escort,0.1,0.1,0.0,4.7%
Personal Business,0.1,0.1,0.0,4.6%
Shop,0.2,0.2,0.0,2.1%
Meal,0.1,0.1,0.0,4.4%
Social,0.3,0.3,-0.0,-0.5%


In [17]:
tours_per_ps_by_purp(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

,Tours per Person (DaysimOutputs),Tours per Person (2023Survey),Difference (Tours per Person (DaysimOutputs) - Tours per Person (2023Survey)),% Difference (Tours per Person (DaysimOutputs) - Tours per Person (2023Survey))
Tour Purpose,,,,
Work,0.3,0.3,0.0,11.7%
School,0.2,0.1,0.0,10.9%
Escort,0.1,0.1,0.0,15.9%
Personal Business,0.1,0.1,0.0,87.9%
Shop,0.1,0.1,0.1,61.2%
Meal,0.1,0.1,0.1,60.0%
Social,0.4,0.3,0.1,16.2%


## Tours per Person by Mode

In [18]:
def tours_per_ps_by_mode(data1=data_daysim, data2=data_survey, tag='PSRC Region'):
    ##Tours per Person by Mode
    Person_1_total = get_total(data1['Person']['psexpfac'])
    Person_2_total = get_total(data2['Person']['psexpfac'])
    tpbp1 = data1['Tour'][['tmodetp','toexpfac']].groupby('tmodetp').sum()['toexpfac'] / Person_1_total
    tpbp2 = data2['Tour_cloned'][['tmodetp','toexpfac']].groupby('tmodetp').sum()['toexpfac'] / Person_2_total
    tpbp = pd.DataFrame()
    tpbp['Tours per Person (DaysimOutputs)'] = tpbp1
    tpbp[f'Tours per Person ({survey_year}Survey)'] = tpbp2
    tpbp = get_differences(tpbp, 'Tours per Person (DaysimOutputs)', 'Tours per Person ({survey_year}Survey)', 2)
    tpbp = recode_index(tpbp, 'tmodetp', 'Tour Mode')
    tpbp = tpbp.loc[mode_cat.values()]
    # table
    display(tpbp.style.format({
        'Tours per Person (DaysimOutputs)': '{:,.1f}',
        f'Tours per Person ({survey_year}Survey)': '{:,.1f}',
        f"Difference (Tours per Person (DaysimOutputs) - Tours per Person ({survey_year}Survey))": '{:,.1f}',
        f"% Difference (Tours per Person (DaysimOutputs) - Tours per Person ({survey_year}Survey))": '{:,.1f}%',
    }))
    # plot
    fig = px.bar(
        tpbp.reset_index(),
        x='Tour Mode',
        y=['Tours per Person (DaysimOutputs)', f'Tours per Person ({survey_year}Survey)'],
        barmode='group',
        title=f'Tours per Person by Mode ({tag})'
    )
    fig.update_layout(yaxis_title='Tours per Person', 
                      xaxis_title='Tour Mode', 
                      xaxis=dict(showgrid=True), 
                      yaxis=dict(showgrid=True))
    fig.show()    

In [19]:
tours_per_ps_by_mode(data1=data_daysim, data2=data_survey, tag='PSRC Region')

,Tours per Person (DaysimOutputs),Tours per Person (2023Survey),Difference (Tours per Person (DaysimOutputs) - Tours per Person (2023Survey)),% Difference (Tours per Person (DaysimOutputs) - Tours per Person (2023Survey))
Tour Mode,,,,
Walk,0.1,0.1,0.0,0.7%
Bike,0.0,0.0,-0.0,-54.4%
SOV,0.5,0.5,0.0,5.8%
HOV2,0.3,0.3,0.0,0.4%
HOV3+,0.3,0.3,0.0,13.2%
Transit Walk Access,0.1,0.1,-0.0,-17.7%
Transit Auto Access,0.0,0.0,-0.0,-48.5%
School Bus,0.0,0.0,-0.0,-0.9%


In [20]:
tours_per_ps_by_purp(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

,Tours per Person (DaysimOutputs),Tours per Person (2023Survey),Difference (Tours per Person (DaysimOutputs) - Tours per Person (2023Survey)),% Difference (Tours per Person (DaysimOutputs) - Tours per Person (2023Survey))
Tour Purpose,,,,
Work,0.3,0.3,0.0,11.7%
School,0.2,0.1,0.0,10.9%
Escort,0.1,0.1,0.0,15.9%
Personal Business,0.1,0.1,0.0,87.9%
Shop,0.1,0.1,0.1,61.2%
Meal,0.1,0.1,0.1,60.0%
Social,0.4,0.3,0.1,16.2%


## Tour Distance by Purpose

In [21]:
def tours_distance_by_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region'):
    ##Average distance by tour purpose
    name1 = 'DaysimOutputs'
    name2 = f'{survey_year}Survey' 
    ##Filter out unreasonable trip/tour lengths
    ##survey data does not include drive to transit trips, remove- how can we do this without referencing max internal zone?    
    tu_hh_1 = data1['Tour'].merge(data1['Household'][['hhno', 'hhtaz']], on='hhno')
    tu_hh_2 = data2['Tour_cloned'].merge(data2['Household'][['hhno', 'hhtaz']], on='hhno')
    tp_hh_1 = data1['Trip'].merge(data1['Household'][['hhno', 'hhtaz']], on='hhno')
    tp_hh_2 = data2['Trip_cloned'].merge(data2['Household'][['hhno', 'hhtaz']], on='hhno')

    tour_ok_1 = tu_hh_1.\
        query('tautodist>0 and tautodist<200')[['hhno', 'pno', 'tour', 'day', 'tautodist', 'toexpfac', 'pdpurp', 'tmodetp', 'tdtaz']].copy(deep=True)
    tour_ok_2 = tu_hh_2.\
        query('tautodist>0 and tautodist<200')[['hhno', 'pno', 'tour', 'day', 'tautodist', 'toexpfac', 'pdpurp', 'tmodetp', 'tdtaz']].copy(deep=True) 
    trip_ok_1 = tp_hh_1.\
        query('travdist>0 and travdist<200')[['hhno', 'pno', 'tour', 'day', 'travdist', 'trexpfac', 'dpurp', 'mode', 'dtaz']].copy(deep=True)
    trip_ok_2 = tp_hh_2.\
        query('travdist>0 and travdist<200')[['hhno', 'pno', 'tour', 'day', 'travdist', 'trexpfac', 'dpurp', 'mode', 'dtaz']] .copy(deep=True)
    
    #Merge tour and trip files
    tourtrip1 = pd.merge(tour_ok_1[['hhno', 'pno', 'tour', 'day', 'tautodist', 'toexpfac', 'pdpurp', 'tmodetp']],
                       trip_ok_1[['hhno', 'pno', 'tour', 'day', 'trexpfac']],
                       on = ['hhno', 'pno', 'tour', 'day'])
    tourtrip2 = pd.merge(tour_ok_2[['hhno', 'pno', 'tour', 'day', 'tautodist', 'toexpfac', 'pdpurp', 'tmodetp']],
                       trip_ok_2[['hhno', 'pno', 'tour', 'day', 'trexpfac']],
                       on = ['hhno', 'pno', 'tour', 'day'])

    #Compute weighted average of trip length grouped by purpose
    triptotal1 = weighted_average(tourtrip1[['tautodist', 'toexpfac', 'pdpurp']], 'tautodist', 'toexpfac', 'pdpurp')
    triptotal2 = weighted_average(tourtrip2[['tautodist', 'toexpfac', 'pdpurp']], 'tautodist', 'toexpfac', 'pdpurp')

    #Create data frame
    atl1 = pd.DataFrame.from_dict({'Average Tour Length (' + name1 + ')' : triptotal1})
    atl2 = pd.DataFrame.from_dict({'Average Tour Length (' + name2 + ')': triptotal2})
    atl = pd.merge(atl1, atl2, 'outer', left_index = True, right_index = True)
    atl = get_differences(atl, 'Average Tour Length (' + name1 + ')', 'Average Tour Length (' + name2 + ')', 2)
    atl = recode_index(atl, 'pdpurp', 'Tour Purpose')  
    atl = atl.loc[pdpurp_cat.values()]
    # display table
    display(atl.style.format({
        f'Average Tour Length ({name1})': '{:,.1f}',
        f'Average Tour Length ({name2})': '{:,.1f}',
        f'Difference (Average Tour Length ({name1}) - Average Tour Length ({name2}))': '{:,.1f}',
        f'% Difference (Average Tour Length ({name1}) - Average Tour Length ({name2}))': '{:,.1f}%'
    }))
    # bar plot
    fig = px.bar(
        atl.reset_index(),
        x='Tour Purpose',
        y=[f'Average Tour Length ({name1})', f'Average Tour Length ({name2})'],
        barmode='group',
        title=f'Average Tour Distance by Purpose ({tag})'
    )
    fig.update_layout(yaxis_title='Average Tour Distance', 
                      xaxis_title='Tour Purpose', 
                      xaxis=dict(showgrid=True), 
                      yaxis=dict(showgrid=True))
    fig.show()

In [22]:
tours_distance_by_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region')

,Average Tour Length (DaysimOutputs),Average Tour Length (2023Survey),Difference (Average Tour Length (DaysimOutputs) - Average Tour Length (2023Survey)),% Difference (Average Tour Length (DaysimOutputs) - Average Tour Length (2023Survey))
Tour Purpose,,,,
Work,11.6,11.8,-0.2,-2.0%
School,4.8,4.6,0.2,5.3%
Escort,7.4,7.1,0.3,4.7%
Personal Business,6.7,8.0,-1.2,-15.7%
Shop,4.8,4.7,0.1,2.0%
Meal,3.6,3.8,-0.1,-3.8%
Social,5.8,6.9,-1.1,-16.3%


In [23]:
tours_distance_by_purp(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

,Average Tour Length (DaysimOutputs),Average Tour Length (2023Survey),Difference (Average Tour Length (DaysimOutputs) - Average Tour Length (2023Survey)),% Difference (Average Tour Length (DaysimOutputs) - Average Tour Length (2023Survey))
Tour Purpose,,,,
Work,9.2,7.0,2.2,31.1%
School,3.9,4.3,-0.3,-8.2%
Escort,6.6,11.2,-4.6,-41.3%
Personal Business,7.1,5.7,1.4,25.1%
Shop,2.7,4.0,-1.3,-32.0%
Meal,2.9,5.3,-2.4,-46.0%
Social,4.9,8.0,-3.1,-39.0%


## Tour Distance by Mode

In [24]:
def tour_distance_by_mode(data1=data_daysim, data2=data_survey, tag='PSRC Region'):
    #Average Distance by Tour Mode
    name1 = 'DaysimOutputs'
    name2 = f'{survey_year}Survey' 
    ##Filter out unreasonable trip/tour lengths
    ##survey data does not include drive to transit trips, remove- how can we do this without referencing max internal zone?    
    tu_hh_1 = data1['Tour'].merge(data1['Household'][['hhno', 'hhtaz']], on='hhno')
    tu_hh_2 = data2['Tour_cloned'].merge(data2['Household'][['hhno', 'hhtaz']], on='hhno')
    tp_hh_1 = data1['Trip'].merge(data1['Household'][['hhno', 'hhtaz']], on='hhno')
    tp_hh_2 = data2['Trip_cloned'].merge(data2['Household'][['hhno', 'hhtaz']], on='hhno')

    tour_ok_1 = tu_hh_1.\
        query('tautodist>0 and tautodist<200')[['hhno', 'pno', 'tour', 'day', 'tautodist', 'toexpfac', 'pdpurp', 'tmodetp', 'tdtaz']].copy(deep=True)
    tour_ok_2 = tu_hh_2.\
        query('tautodist>0 and tautodist<200')[['hhno', 'pno', 'tour', 'day', 'tautodist', 'toexpfac', 'pdpurp', 'tmodetp', 'tdtaz']].copy(deep=True) 
    trip_ok_1 = tp_hh_1.\
        query('travdist>0 and travdist<200')[['hhno', 'pno', 'tour', 'day', 'travdist', 'trexpfac', 'dpurp', 'mode', 'dtaz']].copy(deep=True)
    trip_ok_2 = tp_hh_2.\
        query('travdist>0 and travdist<200')[['hhno', 'pno', 'tour', 'day', 'travdist', 'trexpfac', 'dpurp', 'mode', 'dtaz']] .copy(deep=True)
    
    #Merge tour and trip files
    tourtrip1 = pd.merge(tour_ok_1[['hhno', 'pno', 'tour', 'day', 'tautodist', 'toexpfac', 'pdpurp', 'tmodetp']],
                       trip_ok_1[['hhno', 'pno', 'tour', 'day', 'trexpfac']],
                       on = ['hhno', 'pno', 'tour', 'day'])
    tourtrip2 = pd.merge(tour_ok_2[['hhno', 'pno', 'tour', 'day', 'tautodist', 'toexpfac', 'pdpurp', 'tmodetp']],
                       trip_ok_2[['hhno', 'pno', 'tour', 'day', 'trexpfac']],
                       on = ['hhno', 'pno', 'tour', 'day'])

    #Compute weighted average of trip length grouped by purpose
    triptotal1 = weighted_average(tourtrip1[['tautodist', 'toexpfac', 'tmodetp']], 'tautodist', 'toexpfac', 'tmodetp')
    triptotal2 = weighted_average(tourtrip2[['tautodist', 'toexpfac', 'tmodetp']], 'tautodist', 'toexpfac', 'tmodetp')

    #Create data frame
    atl1 = pd.DataFrame.from_dict({'Average Tour Length (' + name1 + ')' : triptotal1})
    atl2 = pd.DataFrame.from_dict({'Average Tour Length (' + name2 + ')': triptotal2})
    atl = pd.merge(atl1, atl2, 'outer', left_index = True, right_index = True)
    atl = get_differences(atl, 'Average Tour Length (' + name1 + ')', 'Average Tour Length (' + name2 + ')', 2)
    atl = recode_index(atl, 'tmodetp', 'Tour Mode')  
    atl = atl.loc[mode_cat.values()]
    # display table
    display(atl.style.format({
        f'Average Tour Length ({name1})': '{:,.1f}',
        f'Average Tour Length ({name2})': '{:,.1f}',
        f'Difference (Average Tour Length ({name1}) - Average Tour Length ({name2}))': '{:,.1f}',
        f'% Difference (Average Tour Length ({name1}) - Average Tour Length ({name2}))': '{:,.1f}%'
    }))
    # bar plot
    fig = px.bar(
        atl.reset_index(),
        x='Tour Mode',
        y=[f'Average Tour Length ({name1})', f'Average Tour Length ({name2})'],
        barmode='group',
        title=f'Average Tour Distance by Mode ({tag})'
    )
    fig.update_layout(yaxis_title='Average Tour Distance', 
                      xaxis_title='Tour Mode', 
                      xaxis=dict(showgrid=True), 
                      yaxis=dict(showgrid=True))
    fig.show()

In [25]:
tours_distance_by_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region')

,Average Tour Length (DaysimOutputs),Average Tour Length (2023Survey),Difference (Average Tour Length (DaysimOutputs) - Average Tour Length (2023Survey)),% Difference (Average Tour Length (DaysimOutputs) - Average Tour Length (2023Survey))
Tour Purpose,,,,
Work,11.6,11.8,-0.2,-2.0%
School,4.8,4.6,0.2,5.3%
Escort,7.4,7.1,0.3,4.7%
Personal Business,6.7,8.0,-1.2,-15.7%
Shop,4.8,4.7,0.1,2.0%
Meal,3.6,3.8,-0.1,-3.8%
Social,5.8,6.9,-1.1,-16.3%


In [26]:
tours_distance_by_purp(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

,Average Tour Length (DaysimOutputs),Average Tour Length (2023Survey),Difference (Average Tour Length (DaysimOutputs) - Average Tour Length (2023Survey)),% Difference (Average Tour Length (DaysimOutputs) - Average Tour Length (2023Survey))
Tour Purpose,,,,
Work,9.2,7.0,2.2,31.1%
School,3.9,4.3,-0.3,-8.2%
Escort,6.6,11.2,-4.6,-41.3%
Personal Business,7.1,5.7,1.4,25.1%
Shop,2.7,4.0,-1.3,-32.0%
Meal,2.9,5.3,-2.4,-46.0%
Social,4.9,8.0,-3.1,-39.0%


## Tour Travel Time by Purpose

In [27]:
def tours_tt_by_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region'):
    ##Average distance by tour purpose
    name1 = 'DaysimOutputs'
    name2 = f'{survey_year}Survey' 
    ##Filter out unreasonable trip/tour lengths
    ##survey data does not include drive to transit trips, remove- how can we do this without referencing max internal zone?    
    tu_hh_1 = data1['Tour'].merge(data1['Household'][['hhno', 'hhtaz']], on='hhno')
    tu_hh_2 = data2['Tour_cloned'].merge(data2['Household'][['hhno', 'hhtaz']], on='hhno')
    tp_hh_1 = data1['Trip'].merge(data1['Household'][['hhno', 'hhtaz']], on='hhno')
    tp_hh_2 = data2['Trip_cloned'].merge(data2['Household'][['hhno', 'hhtaz']], on='hhno')

    tour_ok_1 = tu_hh_1.\
        query('tautodist>0 and tautodist<200')[['hhno', 'pno', 'tour', 'day', 'tautotime', 'toexpfac', 'pdpurp', 'tmodetp', 'tdtaz']].copy(deep=True)
    tour_ok_2 = tu_hh_2.\
        query('tautodist>0 and tautodist<200')[['hhno', 'pno', 'tour', 'day', 'tautotime', 'toexpfac', 'pdpurp', 'tmodetp', 'tdtaz']].copy(deep=True) 
    trip_ok_1 = tp_hh_1.\
        query('travdist>0 and travdist<200')[['hhno', 'pno', 'tour', 'day', 'travtime', 'trexpfac', 'dpurp', 'mode', 'dtaz']].copy(deep=True)
    trip_ok_2 = tp_hh_2.\
        query('travdist>0 and travdist<200')[['hhno', 'pno', 'tour', 'day', 'travtime', 'trexpfac', 'dpurp', 'mode', 'dtaz']] .copy(deep=True)
    
    #Merge tour and trip files
    tourtrip1 = pd.merge(tour_ok_1[['hhno', 'pno', 'tour', 'day', 'tautotime', 'toexpfac', 'pdpurp', 'tmodetp']],
                       trip_ok_1[['hhno', 'pno', 'tour', 'day', 'trexpfac']],
                       on = ['hhno', 'pno', 'tour', 'day'])
    tourtrip2 = pd.merge(tour_ok_2[['hhno', 'pno', 'tour', 'day', 'tautotime', 'toexpfac', 'pdpurp', 'tmodetp']],
                       trip_ok_2[['hhno', 'pno', 'tour', 'day', 'trexpfac']],
                       on = ['hhno', 'pno', 'tour', 'day'])

    #Compute weighted average of trip length grouped by purpose
    triptotal1 = weighted_average(tourtrip1[['tautotime', 'toexpfac', 'pdpurp']], 'tautotime', 'toexpfac', 'pdpurp')
    triptotal2 = weighted_average(tourtrip2[['tautotime', 'toexpfac', 'pdpurp']], 'tautotime', 'toexpfac', 'pdpurp')

    #Create data frame
    atl1 = pd.DataFrame.from_dict({'Average Tour Travel Time (' + name1 + ')' : triptotal1})
    atl2 = pd.DataFrame.from_dict({'Average Tour Travel Time (' + name2 + ')': triptotal2})
    atl = pd.merge(atl1, atl2, 'outer', left_index = True, right_index = True)
    atl = get_differences(atl, 'Average Tour Travel Time (' + name1 + ')', 'Average Tour Travel Time (' + name2 + ')', 2)
    atl = recode_index(atl, 'pdpurp', 'Tour Purpose')  
    atl.columns.name = 'Travel Time (minutes)'
    atl = atl.loc[pdpurp_cat.values(), :]
    # display table
    display(atl.style.format({
        f'Average Tour Travel Time ({name1})': '{:,.1f}',
        f'Average Tour Travel Time ({name2})': '{:,.1f}',
        f'Difference (Average Tour Travel Time ({name1}) - Average Tour Travel Time ({name2}))': '{:,.1f}',
        f'% Difference (Average Tour Travel Time ({name1}) - Average Tour Travel Time ({name2}))': '{:,.1f}%'
    }))
    # bar plot
    fig = px.bar(
        atl.reset_index(),
        x='Tour Purpose',
        y=[f'Average Tour Travel Time ({name1})', f'Average Tour Travel Time ({name2})'],
        barmode='group',
        title=f'Average Tour Travel Time by Purpose ({tag})'
    )
    fig.update_layout(yaxis_title='Average Tour Travel Time', 
                      xaxis_title='Tour Purpose', 
                      xaxis=dict(showgrid=True), 
                      yaxis=dict(showgrid=True))
    fig.show()

In [28]:
tours_tt_by_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region')

Travel Time (minutes),Average Tour Travel Time (DaysimOutputs),Average Tour Travel Time (2023Survey),Difference (Average Tour Travel Time (DaysimOutputs) - Average Tour Travel Time (2023Survey)),% Difference (Average Tour Travel Time (DaysimOutputs) - Average Tour Travel Time (2023Survey))
Tour Purpose,,,,
Work,29.0,25.3,3.7,14.5%
School,15.2,13.4,1.8,13.5%
Escort,21.2,16.1,5.1,31.6%
Personal Business,20.1,37.6,-17.5,-46.5%
Shop,16.5,14.4,2.1,14.8%
Meal,13.6,16.7,-3.1,-18.8%
Social,17.6,17.4,0.1,0.7%


In [29]:
tours_tt_by_purp(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

Travel Time (minutes),Average Tour Travel Time (DaysimOutputs),Average Tour Travel Time (2023Survey),Difference (Average Tour Travel Time (DaysimOutputs) - Average Tour Travel Time (2023Survey)),% Difference (Average Tour Travel Time (DaysimOutputs) - Average Tour Travel Time (2023Survey))
Tour Purpose,,,,
Work,18.9,17.5,1.5,8.4%
School,8.8,12.1,-3.2,-26.8%
Escort,13.8,23.2,-9.4,-40.7%
Personal Business,15.0,14.1,0.9,6.5%
Shop,6.8,12.2,-5.4,-44.0%
Meal,7.2,13.5,-6.4,-47.0%
Social,10.8,20.6,-9.9,-48.0%


## Tour Travel Time by Mode

In [30]:
def tours_tt_by_mode(data1=data_daysim, data2=data_survey, tag='PSRC Region'):
    ##Average distance by tour purpose
    name1 = 'DaysimOutputs'
    name2 = f'{survey_year}Survey' 
    ##Filter out unreasonable trip/tour lengths
    ##survey data does not include drive to transit trips, remove- how can we do this without referencing max internal zone?    
    tu_hh_1 = data1['Tour'].merge(data1['Household'][['hhno', 'hhtaz']], on='hhno')
    tu_hh_2 = data2['Tour_cloned'].merge(data2['Household'][['hhno', 'hhtaz']], on='hhno')
    tp_hh_1 = data1['Trip'].merge(data1['Household'][['hhno', 'hhtaz']], on='hhno')
    tp_hh_2 = data2['Trip_cloned'].merge(data2['Household'][['hhno', 'hhtaz']], on='hhno')

    tour_ok_1 = tu_hh_1.\
        query('tautodist>0 and tautodist<200')[['hhno', 'pno', 'tour', 'day', 'tautotime', 'toexpfac', 'pdpurp', 'tmodetp', 'tdtaz']].copy(deep=True)
    tour_ok_2 = tu_hh_2.\
        query('tautodist>0 and tautodist<200')[['hhno', 'pno', 'tour', 'day', 'tautotime', 'toexpfac', 'pdpurp', 'tmodetp', 'tdtaz']].copy(deep=True) 
    trip_ok_1 = tp_hh_1.\
        query('travdist>0 and travdist<200')[['hhno', 'pno', 'tour', 'day', 'travtime', 'trexpfac', 'dpurp', 'mode', 'dtaz']].copy(deep=True)
    trip_ok_2 = tp_hh_2.\
        query('travdist>0 and travdist<200')[['hhno', 'pno', 'tour', 'day', 'travtime', 'trexpfac', 'dpurp', 'mode', 'dtaz']] .copy(deep=True)
    
    #Merge tour and trip files
    tourtrip1 = pd.merge(tour_ok_1[['hhno', 'pno', 'tour', 'day', 'tautotime', 'toexpfac', 'tmodetp']],
                       trip_ok_1[['hhno', 'pno', 'tour', 'day', 'trexpfac']],
                       on = ['hhno', 'pno', 'tour', 'day'])
    tourtrip2 = pd.merge(tour_ok_2[['hhno', 'pno', 'tour', 'day', 'tautotime', 'toexpfac', 'tmodetp']],
                       trip_ok_2[['hhno', 'pno', 'tour', 'day', 'trexpfac']],
                       on = ['hhno', 'pno', 'tour', 'day'])

    #Compute weighted average of trip length grouped by purpose
    triptotal1 = weighted_average(tourtrip1[['tautotime', 'toexpfac', 'tmodetp']], 'tautotime', 'toexpfac', 'tmodetp')
    triptotal2 = weighted_average(tourtrip2[['tautotime', 'toexpfac', 'tmodetp']], 'tautotime', 'toexpfac', 'tmodetp')

    #Create data frame
    atl1 = pd.DataFrame.from_dict({'Average Tour Travel Time (' + name1 + ')' : triptotal1})
    atl2 = pd.DataFrame.from_dict({'Average Tour Travel Time (' + name2 + ')': triptotal2})
    atl = pd.merge(atl1, atl2, 'outer', left_index = True, right_index = True)
    atl = get_differences(atl, 'Average Tour Travel Time (' + name1 + ')', 'Average Tour Travel Time (' + name2 + ')', 2)
    atl = recode_index(atl, 'tmodetp', 'Tour Mode')  
    atl.columns.name = 'Travel Time (minutes)'
    atl = atl.loc[mode_cat.values()]
    # display table
    display(atl.style.format({
        f'Average Tour Travel Time ({name1})': '{:,.1f}',
        f'Average Tour Travel Time ({name2})': '{:,.1f}',
        f'Difference (Average Tour Travel Time ({name1}) - Average Tour Travel Time ({name2}))': '{:,.1f}',
        f'% Difference (Average Tour Travel Time ({name1}) - Average Tour Travel Time ({name2}))': '{:,.1f}%'
    }))
    # bar plot
    fig = px.bar(
        atl.reset_index(),
        x='Tour Mode',
        y=[f'Average Tour Travel Time ({name1})', f'Average Tour Travel Time ({name2})'],
        barmode='group',
        title=f'Average Tour Travel Time by Mode ({tag})'
    )
    fig.update_layout(yaxis_title='Average Tour Travel Time', 
                      xaxis_title='Tour Mode', 
                      xaxis=dict(showgrid=True), 
                      yaxis=dict(showgrid=True))
    fig.show()

In [31]:
tours_tt_by_mode(data1=data_daysim, data2=data_survey, tag='PSRC Region')

Travel Time (minutes),Average Tour Travel Time (DaysimOutputs),Average Tour Travel Time (2023Survey),Difference (Average Tour Travel Time (DaysimOutputs) - Average Tour Travel Time (2023Survey)),% Difference (Average Tour Travel Time (DaysimOutputs) - Average Tour Travel Time (2023Survey))
Tour Mode,,,,
Walk,3.6,19.1,-15.4,-81.0%
Bike,9.0,11.9,-2.9,-24.1%
SOV,24.5,19.6,5.0,25.4%
HOV2,22.2,17.7,4.5,25.4%
HOV3+,22.4,17.8,4.6,25.8%
Transit Walk Access,27.1,67.9,-40.8,-60.1%
Transit Auto Access,36.7,37.9,-1.3,-3.4%
School Bus,12.9,10.5,2.4,22.8%


In [32]:
tours_tt_by_mode(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

Travel Time (minutes),Average Tour Travel Time (DaysimOutputs),Average Tour Travel Time (2023Survey),Difference (Average Tour Travel Time (DaysimOutputs) - Average Tour Travel Time (2023Survey)),% Difference (Average Tour Travel Time (DaysimOutputs) - Average Tour Travel Time (2023Survey))
Tour Mode,,,,
Walk,2.7,12.7,-9.9,-78.6%
Bike,7.5,10.2,-2.7,-26.7%
SOV,14.4,14.3,0.1,0.8%
HOV2,13.5,23.7,-10.2,-43.1%
HOV3+,13.2,16.4,-3.2,-19.4%
Transit Walk Access,13.8,22.7,-8.9,-39.2%
Transit Auto Access,15.6,37.5,-21.9,-58.3%
School Bus,6.9,8.7,-1.8,-20.3%


## Subtour Purpose Share

In [33]:
def subtours_by_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region',
                     feature='dpurp', feature_label='Purpose', feature_order=[]):
    name1 = 'DaysimOutputs'
    name2 = f'{survey_year}Survey'
    #Subtour Purpose Share
    tu_hh_1 = data1['Tour'].merge(data1['Household'][['hhno', 'hhtaz']], on='hhno')
    tu_hh_2 = data2['Tour_cloned'].merge(data2['Household'][['hhno', 'hhtaz']], on='hhno')
    tour_ok_1 = tu_hh_1.\
        query('tautodist>0 and tautodist<200')[['hhno', 'pno', 'tour', 'parent', 'day', 'tautodist', 'toexpfac', 'pdpurp', 'tmodetp', 'tdtaz']].copy(deep=True)
    tour_ok_2 = tu_hh_2.\
        query('tautodist>0 and tautodist<200')[['hhno', 'pno', 'tour', 'parent', 'day', 'tautodist', 'toexpfac', 'pdpurp', 'tmodetp', 'tdtaz']].copy(deep=True) 
    Tour_1_total = get_total(data1['Tour']['toexpfac'])
    Tour_2_total = get_total(data2['Tour_cloned']['toexpfac'])
    subtour_ok_1 = tour_ok_1[tour_ok_1['parent']>0].copy(deep=True)
    subtour_ok_2 = tour_ok_2[tour_ok_2['parent']>0].copy(deep=True)
    stpurpose1 = subtour_ok_1[[feature,'toexpfac']].groupby(feature).sum()['toexpfac']
    stpurpose2 = subtour_ok_2[[feature,'toexpfac']].groupby(feature).sum()['toexpfac']
    subpurposeshare1 = stpurpose1 / Tour_1_total * 100
    subpurposeshare2 = stpurpose2 / Tour_2_total * 100
    spsdf = pd.DataFrame()
    difference = subpurposeshare1 - subpurposeshare2
    subpurposeshare1 = subpurposeshare1.sort_index()
    spsdf[name1 + ' Share (%)'] = subpurposeshare1
    spsdf[name1 + ' # of Subtours'] = stpurpose1
    subpurposeshare2 = subpurposeshare2.sort_index()
    spsdf[name2 + ' Share (%)'] = subpurposeshare2
    spsdf[name2 + ' # of Subtours'] = stpurpose2
    spsdf = get_differences(spsdf, name1 + ' Share (%)', name2 + ' Share (%)', 2)
    spsdf = recode_index(spsdf, feature, feature_label)
    spsdf = spsdf.loc[feature_order, [name1 + ' Share (%)',
                                      name1 + ' # of Subtours',
                                      name2 + ' Share (%)',
                                      name2 + ' # of Subtours',
                                      f'Difference ({name1} Share (%) - {name2} Share (%))']]
    spsdf = spsdf.loc[pdpurp_cat.values()]
    # display table
    display(spsdf.style.format({
        f'{name1} Share (%)': '{:,.1f}%',
        f'{name1} # of Subtours': '{:,.0f}',
        f'{name2} Share (%)': '{:,.1f}%',
        f'{name2} # of Subtours': '{:,.0f}',
        f'Difference ({name1} Share (%) - {name2} Share (%))': '{:,.1f}%'
    }))
    # bar plot
    fig = px.bar(
        spsdf.reset_index(),
        x=feature_label,
        y=[f'{name1} Share (%)', f'{name2} Share (%)'],
        barmode='group',
        title=f'Subtour {feature_label} Share ({tag})'
    )
    fig.update_layout(yaxis_title='Share (%)', 
                      xaxis_title=feature_label, 
                      showlegend=True,
                      xaxis=dict(showgrid=True), 
                      yaxis=dict(showgrid=True))
    fig.update_yaxes(ticksuffix='%', gridcolor='lightgrey', showgrid=True)
    fig.show()

In [34]:
subtours_by_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region',
                 feature='pdpurp', feature_label='Purpose', feature_order=pdpurp_cat.values())

,DaysimOutputs Share (%),DaysimOutputs # of Subtours,2023Survey Share (%),2023Survey # of Subtours,Difference (DaysimOutputs Share (%) - 2023Survey Share (%))
Purpose,,,,,
Work,1.4%,"79,619",0.6%,"31,396",0.7%
School,0.0%,98,0.0%,29,0.0%
Escort,0.0%,422,0.0%,409,-0.0%
Personal Business,0.0%,"1,531",0.0%,"2,227",-0.0%
Shop,0.2%,"11,339",0.2%,"8,681",0.0%
Meal,0.9%,"53,792",0.7%,"33,040",0.3%
Social,0.5%,"31,053",0.5%,"27,150",-0.0%


In [35]:
subtours_by_purp(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR',
                     feature='pdpurp', feature_label='Purpose', feature_order=pdpurp_cat.values())

,DaysimOutputs Share (%),DaysimOutputs # of Subtours,2023Survey Share (%),2023Survey # of Subtours,Difference (DaysimOutputs Share (%) - 2023Survey Share (%))
Purpose,,,,,
Work,1.2%,"5,597",1.6%,"5,405",-0.4%
School,0.0%,4,nan%,nan,nan%
Escort,0.0%,25,0.0%,63,-0.0%
Personal Business,0.0%,99,0.3%,"1,071",-0.3%
Shop,0.2%,806,0.1%,217,0.1%
Meal,0.9%,"4,213",2.1%,"7,153",-1.2%
Social,0.5%,"2,091",nan%,nan,nan%


## Number of Stops

In [36]:
def num_stops_all_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region'):
    ## Number of Stops for all Purposes
    name1 = 'DaysimOutputs'
    name2 = f'{survey_year}Survey'
    # number of stops can be retrieved from tripsh1 and tripsh2, which aggregate the number of stops from origin and destination
    data1['Tour']['all_stops'] = data1['Tour']['tripsh1'] + data1['Tour']['tripsh2'] - 2  # minus the origin and destination
    data2['Tour_cloned']['all_stops'] = data2['Tour_cloned']['tripsh1'] + data2['Tour_cloned']['tripsh2'] - 2  # minus the origin and destination
    # calculate the percentage of each number of stops
    intermediate_stops1 = data1['Tour'].groupby('all_stops')['toexpfac'].sum().reset_index()
    intermediate_stops2 = data2['Tour_cloned'].groupby('all_stops')['toexpfac'].sum().reset_index()
    intermediate_stops1['percentage'] = intermediate_stops1['toexpfac'] / intermediate_stops1['toexpfac'].sum() * 100
    intermediate_stops2['percentage'] = intermediate_stops2['toexpfac'] / intermediate_stops2['toexpfac'].sum() * 100
    # compare the first 10 stops
    imstp1 = intermediate_stops1[intermediate_stops1['all_stops']<=10].copy(deep=True)
    imstp2 = intermediate_stops2[intermediate_stops2['all_stops']<=10].copy(deep=True)
    imstp1 = imstp1.set_index('all_stops').reindex(range(0, 11), fill_value=0).reset_index()
    imstp2 = imstp2.set_index('all_stops').reindex(range(0, 11), fill_value=0).reset_index()
    s_all = pd.DataFrame() 
    s_all['% of (' + name1 + ')'] = list(imstp1['percentage'])
    s_all['% of (' + name2 + ')'] = list(imstp2['percentage'])
    s_all['# Stops in tour'] = range(0, 11)
    s_all = s_all.set_index('# Stops in tour')
    s_all = get_differences(s_all, '% of (' + name1 + ')', '% of (' + name2 + ')', 2)
    s_all = s_all[["% of (DaysimOutputs)", f'% of ({survey_year}Survey)', f'Difference (% of ({name1}) - % of ({name2}))']]
    # display table
    display(s_all.style.format({
        f'% of ({name1})': '{:,.1f}%',
        f'% of ({name2})': '{:,.1f}%',
        f'Difference (% of ({name1}) - % of ({name2}))': '{:,.1f}',
    }))
    # bar plot
    fig = px.bar(
        s_all.reset_index(),
        x='# Stops in tour',
        y=[f'% of ({name1})', f'% of ({name2})'],
        barmode='group',
        title=f'Number of Stops per Tour (All Purposes) - {tag}'
    )
    fig.update_layout(yaxis_title='Percentage of Tours', 
                      xaxis_title='Number of Stops', 
                      xaxis=dict(showgrid=True), 
                      yaxis=dict(showgrid=True))
    fig.update_yaxes(ticksuffix='%')
    fig.show()
    

In [37]:
num_stops_all_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region')

,% of (DaysimOutputs),% of (2023Survey),Difference (% of (DaysimOutputs) - % of (2023Survey))
# Stops in tour,,,
0,56.7%,60.1%,-3.4
1,23.4%,21.0%,2.3
2,11.9%,10.5%,1.4
3,4.8%,4.7%,0.1
4,1.9%,1.8%,0.2
5,0.8%,0.8%,-0.1
6,0.3%,0.4%,-0.1
7,0.1%,0.4%,-0.3
8,0.1%,0.3%,-0.2


In [38]:
num_stops_all_purp(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

,% of (DaysimOutputs),% of (2023Survey),Difference (% of (DaysimOutputs) - % of (2023Survey))
# Stops in tour,,,
0,57.3%,63.8%,-6.5
1,23.4%,23.4%,-0.0
2,11.7%,8.8%,2.8
3,4.6%,1.6%,3.1
4,1.9%,1.5%,0.3
5,0.7%,1.0%,-0.3
6,0.3%,0.0%,0.3
7,0.1%,0.0%,0.1
8,0.1%,0.0%,0.1


## Trip per Person

In [39]:
from collections import OrderedDict

def wkbased_subtour_generation(data1=data_daysim, data2=data_survey, tag='PSRC Region'):
    ##Work-Based Subtour Generation
    name1 = 'DaysimOutputs'
    name2 = f'{survey_year}Survey'
    #Total trips per person
    Person_1_total = get_total(data1['Person']['psexpfac'])
    Person_2_total = get_total(data2['Person']['psexpfac'])
    atp1 = get_total(data1['Trip']['trexpfac']) / Person_1_total
    atp2 = get_total(data2['Trip_cloned']['trexpfac']) / Person_2_total
    travdist_data1 = data1['Trip'][(data1['Trip']['travdist']>0)]
    travdist_data1 = travdist_data1[(travdist_data1['travdist']<200)].copy(deep=True)
    travdist_data2 = data2['Trip_cloned'].query('travdist > 0 and travdist < 200').copy(deep=True)
    atl1 = weighted_average(travdist_data1, 'travdist', 'trexpfac')
    atl2 = weighted_average(travdist_data2, 'travdist', 'trexpfac')
    ttp1 = [atp1, atl1]
    ttp2 = [atp2, atl2]
    label = ['Average Trips Per Person', 'Average Trip Length']
    items = OrderedDict((('', label), (name1, ttp1), (name2, ttp2)))
    ttp = pd.DataFrame.from_dict(items)
    ttp = ttp.set_index('')
    ttp = get_differences(ttp, name1, name2, 2)
    # table
    display(ttp.style.format({
        name1: '{:,.1f}',
        name2: '{:,.1f}',
        f'Difference ({name1} - {name2})': '{:,.1f}',
        f'% Difference ({name1} - {name2})': '{:,.1f}%'
    }))

In [40]:
wkbased_subtour_generation(data1=data_daysim, data2=data_survey, tag='PSRC Region')

,DaysimOutputs,2023Survey,Difference (DaysimOutputs - 2023Survey),% Difference (DaysimOutputs - 2023Survey)
,,,,
Average Trips Per Person,3.8,3.7,0.1,2.8%
Average Trip Length,5.8,5.9,-0.0,-0.8%


In [41]:
wkbased_subtour_generation(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

,DaysimOutputs,2023Survey,Difference (DaysimOutputs - 2023Survey),% Difference (DaysimOutputs - 2023Survey)
,,,,
Average Trips Per Person,3.9,3.0,1.0,32.8%
Average Trip Length,4.5,5.9,-1.4,-22.9%


## Trips per Person by Purpose

In [42]:
def trip_rate_by_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region'):
    ##Trip Rates by Purpose
    name1 = 'DaysimOutputs'
    name2 = f'{survey_year}Survey'
    Person_1_total = get_total(data1['Person']['psexpfac'])
    Person_2_total = get_total(data2['Person']['psexpfac'])
    trp1 = data1['Trip'][['dpurp', 'trexpfac']].groupby('dpurp').sum()['trexpfac'] / Person_1_total
    trp2 = data2['Trip_cloned'][['dpurp', 'trexpfac']].groupby('dpurp').sum()['trexpfac'] / Person_2_total
    trp = pd.DataFrame()
    trp['Trips per Person (' + name1 + ')'] = trp1
    trp['Trips per Person (' + name2 + ')'] = trp2
    trp = get_differences(trp, 'Trips per Person (' + name1 + ')', 'Trips per Person (' + name2 + ')', 2)
    trp = recode_index(trp, 'dpurp', 'Destination Purpose')
    trp = trp.loc[pdpurp_cat.values()]
    # table
    display(trp.style.format({
        f'Trips per Person ({name1})': '{:,.1f}',
        f'Trips per Person ({name2})': '{:,.1f}',
        f'Difference (Trips per Person ({name1}) - Trips per Person ({name2}))': '{:,.1f}',
        f'% Difference (Trips per Person ({name1}) - Trips per Person ({name2}))': '{:,.1f}%'
    }))
    fig = px.bar(
        trp.reset_index(),
        x='Destination Purpose',
        y=[f'Trips per Person ({name1})', f'Trips per Person ({name2})'],
        barmode='group',
        title=f'Trip Rates by Purpose ({tag})'
    )
    fig.update_layout(yaxis_title='Trips per Person', 
                      xaxis_title='Destination Purpose',
                      xaxis=dict(showgrid=True), yaxis=dict(showgrid=True))
    
    fig.show()

In [43]:
trip_rate_by_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region')

,Trips per Person (DaysimOutputs),Trips per Person (2023Survey),Difference (Trips per Person (DaysimOutputs) - Trips per Person (2023Survey)),% Difference (Trips per Person (DaysimOutputs) - Trips per Person (2023Survey))
Destination Purpose,,,,
Work,0.5,0.5,-0.0,-0.5%
School,0.1,0.1,0.0,2.8%
Escort,0.4,0.4,0.0,0.5%
Personal Business,0.2,0.2,-0.0,-14.0%
Shop,0.5,0.5,-0.0,-0.5%
Meal,0.2,0.2,0.0,3.9%
Social,0.5,0.5,0.1,9.4%


In [44]:
trip_rate_by_purp(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

,Trips per Person (DaysimOutputs),Trips per Person (2023Survey),Difference (Trips per Person (DaysimOutputs) - Trips per Person (2023Survey)),% Difference (Trips per Person (DaysimOutputs) - Trips per Person (2023Survey))
Destination Purpose,,,,
Work,0.5,0.4,0.1,24.4%
School,0.2,0.1,0.0,16.5%
Escort,0.4,0.3,0.1,16.4%
Personal Business,0.2,0.1,0.0,21.8%
Shop,0.5,0.2,0.2,110.1%
Meal,0.3,0.3,0.0,9.9%
Social,0.6,0.5,0.2,37.0%


## Trips per Person by Mode

In [45]:
def trip_rate_by_mode(data1=data_daysim, data2=data_survey, tag='PSRC Region'):
    ##Trip Rates by Mode
    name1 = 'DaysimOutputs'
    name2 = f'{survey_year}Survey'
    Person_1_total = get_total(data1['Person']['psexpfac'])
    Person_2_total = get_total(data2['Person']['psexpfac'])
    trp1 = data1['Trip'][['mode', 'trexpfac']].groupby('mode').sum()['trexpfac'] / Person_1_total
    trp2 = data2['Trip_cloned'][['mode', 'trexpfac']].groupby('mode').sum()['trexpfac'] / Person_2_total
    trp = pd.DataFrame()
    trp['Trips per Person (' + name1 + ')'] = trp1
    trp['Trips per Person (' + name2 + ')'] = trp2
    trp = get_differences(trp, 'Trips per Person (' + name1 + ')', 'Trips per Person (' + name2 + ')', 2)
    trp = recode_index(trp, 'mode', 'Destination Mode')
    trp = trp.loc[trip_mode_cat.values()]
    # table
    display(trp.style.format({
        f'Trips per Person ({name1})': '{:,.1f}',
        f'Trips per Person ({name2})': '{:,.1f}',
        f'Difference (Trips per Person ({name1}) - Trips per Person ({name2}))': '{:,.1f}',
        f'% Difference (Trips per Person ({name1}) - Trips per Person ({name2}))': '{:,.1f}%'
    }))
    fig = px.bar(
        trp.reset_index(),
        x='Destination Mode',
        y=[f'Trips per Person ({name1})', f'Trips per Person ({name2})'],
        barmode='group',
        title=f'Trip Rates by Mode ({tag})'
    )
    fig.update_layout(yaxis_title='Trips per Person', 
                      xaxis_title='Number of Stops', 
                      xaxis=dict(showgrid=True), 
                      yaxis=dict(showgrid=True))
    fig.show()

In [46]:
trip_rate_by_mode(data1=data_daysim, data2=data_survey, tag='PSRC Region')

,Trips per Person (DaysimOutputs),Trips per Person (2023Survey),Difference (Trips per Person (DaysimOutputs) - Trips per Person (2023Survey)),% Difference (Trips per Person (DaysimOutputs) - Trips per Person (2023Survey))
Destination Mode,,,,
Walk,0.6,0.4,0.1,37.1%
Bike,0.0,0.0,-0.0,-80.3%
SOV,1.8,1.7,0.1,8.9%
HOV2,0.9,0.8,0.1,13.8%
HOV3+,0.4,0.6,-0.2,-28.7%
Transit,0.1,0.1,-0.0,-40.6%
School Bus,0.0,0.1,-0.0,-30.3%


In [47]:
trip_rate_by_mode(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

,Trips per Person (DaysimOutputs),Trips per Person (2023Survey),Difference (Trips per Person (DaysimOutputs) - Trips per Person (2023Survey)),% Difference (Trips per Person (DaysimOutputs) - Trips per Person (2023Survey))
Destination Mode,,,,
Walk,0.5,0.4,0.1,18.4%
Bike,0.0,0.0,-0.0,-73.3%
SOV,1.9,1.2,0.7,58.7%
HOV2,1.0,0.7,0.3,38.6%
HOV3+,0.4,0.4,0.0,9.2%
Transit,0.1,0.1,-0.0,-17.5%
School Bus,0.1,0.1,-0.0,-26.6%


## Trip Distance by Purpose

In [48]:
def trips_distance_by_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region'):
    ##Average distance by tour purpose
    name1 = 'DaysimOutputs'
    name2 = f'{survey_year}Survey' 
    ##Filter out unreasonable trip/tour lengths
    ##survey data does not include drive to transit trips, remove- how can we do this without referencing max internal zone?    
    tp_hh_1 = data1['Trip'].merge(data1['Household'][['hhno', 'hhtaz']], on='hhno')
    tp_hh_2 = data2['Trip_cloned'].merge(data2['Household'][['hhno', 'hhtaz']], on='hhno')

    trip_ok_1 = tp_hh_1.\
        query('travdist>0 and travdist<200')[['hhno', 'pno', 'tour', 'day', 'travdist', 'trexpfac', 'dpurp', 'mode', 'dtaz']].copy(deep=True)
    trip_ok_2 = tp_hh_2.\
        query('travdist>0 and travdist<200')[['hhno', 'pno', 'tour', 'day', 'travdist', 'trexpfac', 'dpurp', 'mode', 'dtaz']] .copy(deep=True)

    #Compute weighted average of trip length grouped by purpose
    triptotal1 = weighted_average(trip_ok_1[['travdist', 'trexpfac', 'dpurp']], 'travdist', 'trexpfac', 'dpurp')
    triptotal2 = weighted_average(trip_ok_2[['travdist', 'trexpfac', 'dpurp']], 'travdist', 'trexpfac', 'dpurp')

    #Create data frame
    atl1 = pd.DataFrame.from_dict({'Average Trip Length (' + name1 + ')' : triptotal1})
    atl2 = pd.DataFrame.from_dict({'Average Trip Length (' + name2 + ')': triptotal2})
    atl = pd.merge(atl1, atl2, 'outer', left_index = True, right_index = True)
    atl = get_differences(atl, 'Average Trip Length (' + name1 + ')', 'Average Trip Length (' + name2 + ')', 2)
    atl = recode_index(atl, 'dpurp', 'Trip Purpose') 
    atl = atl.loc[pdpurp_cat.values()] 
    # display table
    display(atl.style.format({
        f'Average Trip Length ({name1})': '{:,.1f}',
        f'Average Trip Length ({name2})': '{:,.1f}',
        f'Difference (Average Trip Length ({name1}) - Average Trip Length ({name2}))': '{:,.1f}',
        f'% Difference (Average Trip Length ({name1}) - Average Trip Length ({name2}))': '{:,.1f}%'
    }))
    # bar plot
    fig = px.bar(
        atl.reset_index(),
        x='Trip Purpose',
        y=[f'Average Trip Length ({name1})', f'Average Trip Length ({name2})'],
        barmode='group',
        title=f'Average Trip Distance by Purpose ({tag})'
    )
    fig.update_layout(yaxis_title='Average Trip Distance', 
                      xaxis_title='Trip Purpose', 
                      xaxis=dict(showgrid=True), 
                      yaxis=dict(showgrid=True))
    fig.show()

In [49]:
trips_distance_by_purp(data1=data_daysim, data2=data_survey, tag='PSRC Region')

,Average Trip Length (DaysimOutputs),Average Trip Length (2023Survey),Difference (Average Trip Length (DaysimOutputs) - Average Trip Length (2023Survey)),% Difference (Average Trip Length (DaysimOutputs) - Average Trip Length (2023Survey))
Trip Purpose,,,,
Work,9.0,9.2,-0.2,-2.4%
School,4.7,3.9,0.8,19.2%
Escort,5.9,5.7,0.2,2.9%
Personal Business,5.3,5.6,-0.3,-6.2%
Shop,4.3,4.4,-0.0,-0.9%
Meal,3.8,3.3,0.5,14.7%
Social,5.2,6.7,-1.5,-22.0%


In [50]:
trips_distance_by_purp(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

,Average Trip Length (DaysimOutputs),Average Trip Length (2023Survey),Difference (Average Trip Length (DaysimOutputs) - Average Trip Length (2023Survey)),% Difference (Average Trip Length (DaysimOutputs) - Average Trip Length (2023Survey))
Trip Purpose,,,,
Work,7.6,6.7,0.9,13.2%
School,3.8,4.2,-0.4,-9.6%
Escort,4.8,7.5,-2.7,-36.3%
Personal Business,5.0,4.2,0.8,18.5%
Shop,3.0,4.2,-1.1,-27.3%
Meal,3.1,2.9,0.2,7.4%
Social,4.4,6.4,-2.0,-31.8%


## Trip Distance by Mode

In [51]:
def trips_distance_by_mode(data1=data_daysim, data2=data_survey, tag='PSRC Region'):
    ##Average distance by tour purpose
    name1 = 'DaysimOutputs'
    name2 = f'{survey_year}Survey' 
    ##Filter out unreasonable trip/tour lengths
    ##survey data does not include drive to transit trips, remove- how can we do this without referencing max internal zone?    
    tp_hh_1 = data1['Trip'].merge(data1['Household'][['hhno', 'hhtaz']], on='hhno')
    tp_hh_2 = data2['Trip_cloned'].merge(data2['Household'][['hhno', 'hhtaz']], on='hhno')

    trip_ok_1 = tp_hh_1.\
        query('travdist>0 and travdist<200')[['hhno', 'pno', 'tour', 'day', 'travdist', 'trexpfac', 'dpurp', 'mode', 'dtaz']].copy(deep=True)
    trip_ok_2 = tp_hh_2.\
        query('travdist>0 and travdist<200')[['hhno', 'pno', 'tour', 'day', 'travdist', 'trexpfac', 'dpurp', 'mode', 'dtaz']] .copy(deep=True)

    #Compute weighted average of trip length grouped by purpose
    triptotal1 = weighted_average(trip_ok_1[['travdist', 'trexpfac', 'mode']], 'travdist', 'trexpfac', 'mode')
    triptotal2 = weighted_average(trip_ok_2[['travdist', 'trexpfac', 'mode']], 'travdist', 'trexpfac', 'mode')

    #Create data frame
    atl1 = pd.DataFrame.from_dict({'Average Trip Length (' + name1 + ')' : triptotal1})
    atl2 = pd.DataFrame.from_dict({'Average Trip Length (' + name2 + ')': triptotal2})
    atl = pd.merge(atl1, atl2, 'outer', left_index = True, right_index = True)
    atl = get_differences(atl, 'Average Trip Length (' + name1 + ')', 'Average Trip Length (' + name2 + ')', 2)
    atl = recode_index(atl, 'mode', 'Trip Purpose')  
    atl = atl.loc[trip_mode_cat.values()]
    # display table
    display(atl.style.format({
        f'Average Trip Length ({name1})': '{:,.1f}',
        f'Average Trip Length ({name2})': '{:,.1f}',
        f'Difference (Average Trip Length ({name1}) - Average Trip Length ({name2}))': '{:,.1f}',
        f'% Difference (Average Trip Length ({name1}) - Average Trip Length ({name2}))': '{:,.1f}%'
    }))
    # bar plot
    fig = px.bar(
        atl.reset_index(),
        x='Trip Purpose',
        y=[f'Average Trip Length ({name1})', f'Average Trip Length ({name2})'],
        barmode='group',
        title=f'Average Trip Distance by Purpose ({tag})'
    )
    fig.update_layout(yaxis_title='Average Trip Distance', 
                      xaxis_title='Trip Purpose', 
                      xaxis=dict(showgrid=True), 
                      yaxis=dict(showgrid=True))
    fig.show()

In [52]:
trips_distance_by_mode(data1=data_daysim, data2=data_survey, tag='PSRC Region')

,Average Trip Length (DaysimOutputs),Average Trip Length (2023Survey),Difference (Average Trip Length (DaysimOutputs) - Average Trip Length (2023Survey)),% Difference (Average Trip Length (DaysimOutputs) - Average Trip Length (2023Survey))
Trip Purpose,,,,
Walk,0.9,0.9,-0.0,-0.3%
Bike,4.6,2.0,2.6,130.4%
SOV,7.0,6.9,0.1,1.5%
HOV2,5.9,6.2,-0.3,-5.2%
HOV3+,6.8,7.3,-0.6,-7.7%
Transit,9.7,8.9,0.8,9.4%
School Bus,3.8,3.6,0.2,5.2%


In [53]:
trips_distance_by_mode(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR')

,Average Trip Length (DaysimOutputs),Average Trip Length (2023Survey),Difference (Average Trip Length (DaysimOutputs) - Average Trip Length (2023Survey)),% Difference (Average Trip Length (DaysimOutputs) - Average Trip Length (2023Survey))
Trip Purpose,,,,
Walk,1.4,0.7,0.7,100.6%
Bike,7.5,1.5,6.0,407.0%
SOV,5.4,6.2,-0.8,-12.9%
HOV2,4.5,8.5,-4.0,-47.5%
HOV3+,4.5,5.4,-0.9,-16.5%
Transit,5.5,10.2,-4.7,-46.1%
School Bus,2.8,3.1,-0.4,-11.6%


## Trips per Tour by Tour Purpose

In [54]:
def trips_per_tour_by_tour_feature(data1=data_daysim, data2=data_survey, tag='PSRC Region',
                                   feature='pdpurp', feature_label='Purpose', feature_order=[]):
    ##Average distance by tour purpose
    name1 = 'DaysimOutputs'
    name2 = f'{survey_year}Survey' 
    
    ##Count number of trips on each tour by tour purpose``
    tu_hh_1 = data1['Tour'].merge(data1['Household'][['hhno', 'hhtaz']], on='hhno')
    tu_hh_2 = data2['Tour_cloned'].merge(data2['Household'][['hhno', 'hhtaz']], on='hhno')
    tp_hh_1 = data1['Trip'].merge(data1['Household'][['hhno', 'hhtaz']], on='hhno')
    tp_hh_2 = data2['Trip_cloned'].merge(data2['Household'][['hhno', 'hhtaz']], on='hhno')
    tour_ok_1 = tu_hh_1.\
        query('tautodist>0 and tautodist<200')[['hhno', 'pno', 'tour', 'day', 'tautodist', 'toexpfac', 'pdpurp', 'tmodetp', 'tdtaz']].copy(deep=True)
    tour_ok_2 = tu_hh_2.\
        query('tautodist>0 and tautodist<200')[['hhno', 'pno', 'tour', 'day', 'tautodist', 'toexpfac', 'pdpurp', 'tmodetp', 'tdtaz']].copy(deep=True) 
    trip_ok_1 = tp_hh_1.\
        query('travdist>0 and travdist<200')[['hhno', 'pno', 'tour', 'day', 'travdist', 'trexpfac', 'dpurp', 'mode', 'dtaz']].copy(deep=True)
    trip_ok_2 = tp_hh_2.\
        query('travdist>0 and travdist<200')[['hhno', 'pno', 'tour', 'day', 'travdist', 'trexpfac', 'dpurp', 'mode', 'dtaz']] .copy(deep=True)
    #Merge tour and trip files
    tourtrip1 = pd.merge(tour_ok_1[['hhno', 'pno', 'tour', 'day', 'tautodist', 'toexpfac', 'pdpurp', 'tmodetp']],
                       trip_ok_1[['hhno', 'pno', 'tour', 'day', 'trexpfac']],
                       on = ['hhno', 'pno', 'tour', 'day'])
    tourtrip2 = pd.merge(tour_ok_2[['hhno', 'pno', 'tour', 'day', 'tautodist', 'toexpfac', 'pdpurp', 'tmodetp']],
                       trip_ok_2[['hhno', 'pno', 'tour', 'day', 'trexpfac']],
                       on = ['hhno', 'pno', 'tour', 'day'])
    notrips1 = tourtrip1[['hhno', 'pno', 'tour', 'day', 'trexpfac']].groupby(['hhno', 'pno', 'tour', 'day']).count()['trexpfac']
    notrips2 = tourtrip2[['hhno', 'pno', 'tour', 'day', 'trexpfac']].groupby(['hhno', 'pno', 'tour', 'day']).count()['trexpfac']
    notrips1 = notrips1.reset_index()
    notrips2 = notrips2.reset_index()
    notrips1 = notrips1.rename(columns = {'trexpfac':'notrips'})
    notrips2 = notrips2.rename(columns = {'trexpfac':'notrips'})

    #Merge number of trips with the tour file 
    toursnotrips1 = pd.merge(tour_ok_1[['toexpfac', 'pdpurp', 'hhno', 'pno', 'tour', 'tmodetp']], notrips1, on = ['hhno', 'pno', 'tour'])
    toursnotrips2 = pd.merge(tour_ok_2[['toexpfac', 'pdpurp', 'hhno', 'pno', 'tour', 'tmodetp']], notrips2, on = ['hhno', 'pno', 'tour'])

    #Get the average number of trips per tour
    tourtotal1 = weighted_average(toursnotrips1, 'notrips', 'toexpfac', feature)
    tourtotal2 = weighted_average(toursnotrips2, 'notrips', 'toexpfac', feature)

    #Create data frame
    nttp1 = pd.DataFrame.from_dict({'Avg # Trips/Tour (' + name1 + ')': tourtotal1})
    nttp2 = pd.DataFrame.from_dict({'Avg # Trips/Tour (' + name2 + ')': tourtotal2})
    nttp = pd.merge(nttp1, nttp2, 'outer', left_index = True, right_index = True)
    nttp = get_differences(nttp, 'Avg # Trips/Tour (' + name1 + ')', 'Avg # Trips/Tour (' + name2 + ')', 1)
    nttp = recode_index(nttp, feature, f'Tour {feature_label}')
    nttp = nttp.loc[feature_order]

    # display table
    display(nttp.style.format({
        f'Avg # Trips/Tour ({name1})': '{:,.1f}',
        f'Avg # Trips/Tour ({name2})': '{:,.1f}',
        f'Difference (Avg # Trips/Tour ({name1}) - Avg # Trips/Tour ({name2}))': '{:,.1f}',
        f'% Difference (Avg # Trips/Tour ({name1}) - Avg # Trips/Tour ({name2}))': '{:,.1f}%',
    }))
    # bar plot
    fig = px.bar(
        nttp.reset_index(),
        x=f'Tour {feature_label}',
        y=[f'Avg # Trips/Tour ({name1})', f'Avg # Trips/Tour ({name2})'],
        barmode='group',
        title=f'Average Number of Trips per Tour by Tour {feature_label} ({tag})'
    )
    fig.update_layout(yaxis_title='Average # Trips/Tour', 
                      xaxis_title=f'Tour {feature_label}',
                      xaxis=dict(showgrid=True), 
                      yaxis=dict(showgrid=True))
    fig.show()

In [55]:
trips_per_tour_by_tour_feature(data1=data_daysim, data2=data_survey, tag='PSRC Region',
                               feature='pdpurp', feature_label='Purpose', feature_order=pdpurp_cat.values())

,Avg # Trips/Tour (DaysimOutputs),Avg # Trips/Tour (2023Survey),Difference (Avg # Trips/Tour (DaysimOutputs) - Avg # Trips/Tour (2023Survey)),% Difference (Avg # Trips/Tour (DaysimOutputs) - Avg # Trips/Tour (2023Survey))
Tour Purpose,,,,
Work,3.4,2.9,0.4,14.5%
School,2.8,2.4,0.4,15.9%
Escort,2.6,2.6,-0.0,-1.2%
Personal Business,2.6,2.9,-0.3,-10.4%
Shop,2.7,2.7,-0.0,-1.0%
Meal,2.6,2.6,-0.1,-3.1%
Social,2.4,2.7,-0.3,-11.6%


In [56]:
trips_per_tour_by_tour_feature(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR',
                               feature='pdpurp', feature_label='Purpose', feature_order=pdpurp_cat.values())

,Avg # Trips/Tour (DaysimOutputs),Avg # Trips/Tour (2023Survey),Difference (Avg # Trips/Tour (DaysimOutputs) - Avg # Trips/Tour (2023Survey)),% Difference (Avg # Trips/Tour (DaysimOutputs) - Avg # Trips/Tour (2023Survey))
Tour Purpose,,,,
Work,3.3,2.8,0.5,17.1%
School,2.8,2.1,0.7,32.9%
Escort,2.6,2.4,0.2,8.9%
Personal Business,2.6,2.4,0.1,6.1%
Shop,2.8,2.5,0.3,12.9%
Meal,2.6,2.2,0.4,16.4%
Social,2.4,2.7,-0.3,-12.3%


## Trips per Tour by Tour Mode

In [57]:
trips_per_tour_by_tour_feature(data1=data_daysim, data2=data_survey, tag='PSRC Region',
                               feature='tmodetp', feature_label='Mode', feature_order=mode_cat.values())

,Avg # Trips/Tour (DaysimOutputs),Avg # Trips/Tour (2023Survey),Difference (Avg # Trips/Tour (DaysimOutputs) - Avg # Trips/Tour (2023Survey)),% Difference (Avg # Trips/Tour (DaysimOutputs) - Avg # Trips/Tour (2023Survey))
Tour Mode,,,,
Walk,2.4,2.2,0.2,9.6%
Bike,2.4,2.4,-0.0,-1.8%
SOV,2.8,2.7,0.0,0.9%
HOV2,2.9,2.8,0.1,1.8%
HOV3+,2.9,3.0,-0.1,-2.1%
Transit Walk Access,2.8,2.4,0.3,13.1%
Transit Auto Access,4.8,2.7,2.1,80.8%
School Bus,2.6,2.2,0.4,18.3%


In [58]:
trips_per_tour_by_tour_feature(data1=data_daysim_bkr, data2=data_survey_bkr, tag='BKR',
                               feature='tmodetp', feature_label='Mode', feature_order=mode_cat.values())

,Avg # Trips/Tour (DaysimOutputs),Avg # Trips/Tour (2023Survey),Difference (Avg # Trips/Tour (DaysimOutputs) - Avg # Trips/Tour (2023Survey)),% Difference (Avg # Trips/Tour (DaysimOutputs) - Avg # Trips/Tour (2023Survey))
Tour Mode,,,,
Walk,2.4,2.3,0.1,6.5%
Bike,2.4,2.1,0.3,13.6%
SOV,2.7,2.5,0.2,7.6%
HOV2,2.8,2.8,0.0,0.6%
HOV3+,2.9,2.8,0.1,3.9%
Transit Walk Access,2.7,2.1,0.6,28.3%
Transit Auto Access,4.7,2.4,2.3,96.0%
School Bus,2.6,2.0,0.6,27.7%
